# Infer-17 — Filtre de Kalman : systèmes dynamiques linéaires gaussiens

> **Série Infer.NET (25).** Ce notebook étend [Infer-11 (Séquences / HMM)](Infer-14-Sequences.ipynb)
> au cas où l'état caché est **continu**. Là où le HMM suivait une météo ou un mot
> discret, le **filtre de Kalman** (Kalman, 1960) suit une position, une température,
> un prix — toute grandeur qui évolue continûment avec du bruit. C'est l'algorithme
> d'estimation le plus utilisé au monde (navigation GPS, fusion de capteurs, contrôle,
> finance).

**Durée estimée** : 55 minutes
**Prérequis** : [Infer-11 (Séquences / HMM)](Infer-14-Sequences.ipynb), [Infer-2 (Gaussian Mixtures)](Infer-2-Gaussian-Mixtures.ipynb)

## Objectifs

- Comprendre le **filtre de Kalman** comme l'analogue à état continu du HMM
- Formuler un **système dynamique linéaire gaussien** (prédiction + observation)
- Implémenter la **récursion de filtrage bayésien** via Infer.NET (conjugaison gaussienne)
- **Mesurer** l'apport du filtre : MSE filtrée < MSE brute, variance postérieure bornée


## 1. Motivation : le HMM à état continu

[Infer-11](Infer-14-Sequences.ipynb) modélisait des séquences où l'état caché était
**discret** (quel temps, quel mot de la phrase). Mais de nombreux systèmes suivent une
grandeur **continue** : position d'un mobile, température d'un four, prix d'un actif.
Le **filtre de Kalman** est l'équivalent exact du HMM pour l'état continu gaussien —
le **système dynamique linéaire gaussien** (Linear Dynamical System, LDS).

L'état caché $x_t$ évolue linéairement avec un bruit gaussien (la *dynamique*), et on
l'observe à travers un autre bruit gaussien (le *capteur*) :

$$x_t = x_{t-1} + d + \mathcal{N}(0, Q) \quad \text{(équation de transition)}$$
$$y_t = x_t + \mathcal{N}(0, R) \quad \text{(équation d'observation)}$$

Parce que tout est **gaussien et linéaire**, l'inférence reste **exactement conjugée** :
le postérieur est gaussien à chaque pas, calculable en temps fermé. C'est le cas d'école
où Infer.NET (EP sur un modèle linéaire gaussien) résout l'inférence **de manière exacte**.


In [1]:
#r "nuget: Microsoft.ML.Probabilistic"
#r "nuget: Microsoft.ML.Probabilistic.Compiler"
// restore Infer.NET -- isole dans sa propre cellule (convention de la serie)


The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages Microsoft.ML.Probabilistic, 0.4.2504.701 Microsoft.ML.Probabilistic.Compiler, 0.4.2504.701

Configuration des espaces de noms Infer.NET. Le moteur d'inférence **EP** (Expectation
Propagation) résout la conjugaison gaussienne du filtre de Kalman de manière exacte.
On définit aussi le helper de tirage gaussien (Box-Muller) utilisé pour générer la
vraie trajectoire.


In [2]:
using Microsoft.ML.Probabilistic;
using Microsoft.ML.Probabilistic.Distributions;
using Microsoft.ML.Probabilistic.Models;
using System;
using System.Linq;

// Helper : tirage N(0, 1) par Box-Muller (defini AVANT usage).
double Randn(Random r) {
    double u1 = 1.0 - r.NextDouble();
    double u2 = 1.0 - r.NextDouble();
    return Math.Sqrt(-2.0 * Math.Log(u1)) * Math.Sin(2.0 * Math.PI * u2);
}

// --- Terrain de jeu : vraie trajectoire + observations bruitees ---
// On suit un mobile qui derive lineairement (vitesse d constante + bruit de process Q).
Random rng = new Random(42);
int T = 50;
double drift = 0.3;        // d : vitesse constante (terme de pente)
double processVar = 0.5;   // Q : bruit de dynamique (incertitude sur l'evolution)
double obsVar = 4.0;       // R : bruit de capteur (observations TRES bruitees, R >> Q)

// Vraie trajectoire cachee (INCONNUE du modele -- ground truth pour la validation)
double[] trueState = new double[T];
trueState[0] = 0.0;
for (int t = 1; t < T; t++)
    trueState[t] = trueState[t - 1] + drift + Math.Sqrt(processVar) * Randn(rng);

// Observations bruitees (les SEULES donnees vues par le filtre)
double[] obs = new double[T];
for (int t = 0; t < T; t++)
    obs[t] = trueState[t] + Math.Sqrt(obsVar) * Randn(rng);

Console.WriteLine($"Trajectoire generee : T={T} pas, drift={drift}, Q={processVar}, R={obsVar}");
Console.WriteLine($"R >> Q : le capteur est tres bruite -> le filtre a beaucoup de marge pour aider.");
Console.WriteLine($"5 premieres observations : {string.Join(", ", obs.Take(5).Select(v => v.ToString("F2")))}");


Trajectoire generee : T=50 pas, drift=0,3, Q=0,5, R=4


R >> Q : le capteur est tres bruite -> le filtre a beaucoup de marge pour aider.


5 premieres observations : -4,36, -3,20, -0,44, 1,49, -0,75


## 2. La récursion de Kalman = filtrage bayésien pas-à-pas

Le filtre de Kalman est la **récurrence bayésienne** sur l'état continu. À chaque pas :

1. **Prédire** — propager le postérieur précédent $p(x_{t-1} \mid y_{1:t-1})$ à travers la
   dynamique : la moyenne avance de $d$, la variance augmente de $Q$ (l'incertitude croît).
2. **Mettre à jour** — combiner cet *a priori* avec la nouvelle observation $y_t$ par la
   règle de Bayes : le postérieur $p(x_t \mid y_{1:t})$ a une variance **réduite**.

Comme tout est gaussien-linéaire, chaque pas se réduit à une **conjugaison gaussienne**.
On l'exprime comme un **mini-modèle à deux variables** ($x_t$ latente, $y_t$ observée) et
on laisse Infer.NET inférer le postérieur — qui devient l'a priori du pas suivant.

**Optimisation (pattern C118)** : on **compile une fois** le mini-modèle, avec la moyenne
et la variance de l'a priori comme variables `ObservedValue`. À chaque pas on ne fait que
mettre à jour ces valeurs observées puis ré-inférer — ~0,2 ms par pas au lieu de
recompiler (250 ms).


In [3]:
// --- Recursion du filtre de Kalman via Infer.NET (EP, conjugaison gaussienne) ---
InferenceEngine engine = new InferenceEngine();
engine.ShowProgress = false;

// Modele lineaire gaussien compile UNE SEULE FOIS :
//   x  ~ N(priorMean, priorVar)        [latent, a priori du pas courant]
//   y  ~ N(x, obsVar)                  [observation bruitee de x]
// priorMean et priorVar sont des ObservedValue -> mis a jour chaque pas sans recompiler.
Variable<double> priorMean = Variable.New<double>().Named("priorMean");
Variable<double> priorVar  = Variable.New<double>().Named("priorVar");
Variable<double> x = Variable.GaussianFromMeanAndVariance(priorMean, priorVar).Named("x");
Variable<double> y = Variable.GaussianFromMeanAndVariance(x, obsVar).Named("y");

double[] filtMean = new double[T];
double[] filtVar  = new double[T];

// A priori initial : on initialise sur la 1re observation avec une grande incertitude.
double curMean = obs[0];
double curVar  = obsVar * 4.0;   // incertitude initiale large (on ne sait pas ou est le mobile)

for (int t = 0; t < T; t++) {
    try {
        // 1. PREDIRE : propagation de la dynamique (moyenne += drift, variance += Q)
        double predMean = curMean + drift;
        double predVar  = curVar + processVar;

        // 2. METTRE A JOUR : EP infere le postérieur exact de x sachant y = obs[t]
        priorMean.ObservedValue = predMean;
        priorVar.ObservedValue  = predVar;
        y.ObservedValue         = obs[t];
        Gaussian post = engine.Infer<Gaussian>(x);
        curMean = post.GetMean();
        curVar  = post.GetVariance();
    } catch (Exception ex) {
        Console.WriteLine($"EXCEPTION pas {t}: {ex.Message}");
        curMean = obs[t]; curVar = obsVar;
    }
    filtMean[t] = curMean;
    filtVar[t]  = curVar;
}
Console.WriteLine("Filtrage termine : 50 pas infères par EP (conjugaison gaussienne exacte).");
Console.WriteLine($"Variance postérieure finale : {filtVar[T - 1]:F3}   (var d'observation R = {obsVar:F1})");
Console.WriteLine($"Variance postérieure moyenne : {filtVar.Average():F3}");


Filtrage termine : 50 pas infères par EP (conjugaison gaussienne exacte).


Variance postérieure finale : 1,186   (var d'observation R = 4,0)


Variance postérieure moyenne : 1,254


## 3. Visualisation : trajectoire vraie, observations brutes, estimation filtrée

Le graphe ASCII ci-dessous aligne, à chaque pas de temps (un pas sur deux pour la
lisibilité), les trois séries. Les **observations** (`.`) oscillent fortement autour de
la vraie trajectoire (`|`) à cause du capteur bruité ($R = 4$). L'**estimation filtrée**
(`#`, espérance du postérieur) lisse ce bruit et suit fidèlement la trajectoire cachée.


In [4]:
// --- Visualisation ASCII + metriques honnetes ---
// Axe horizontal = position. On cale l'echelle sur l'ensemble des series.
double lo = Math.Min(trueState.Min(), obs.Min()) - 2;
double hi = Math.Max(trueState.Max(), obs.Max()) + 2;
int W = 60;

string Bar(double v, char c) {
    int n = (int)Math.Round((v - lo) / (hi - lo) * W);
    if (n < 0) n = 0; if (n > W) n = W;
    return new string(c, n);
}

Console.WriteLine($"Echelle : {lo:F1} (gauche) -> {hi:F1} (droite), {W} colonnes");
Console.WriteLine("  VRAI = |   OBS = .   FILT = #");
for (int t = 0; t < T; t += 2) {
    Console.WriteLine($"t={t,2} VRAI {Bar(trueState[t], '|')}");
    Console.WriteLine($"     OBS  {Bar(obs[t], '.')}");
    Console.WriteLine($"     FILT {Bar(filtMean[t], '#')}  (+-{Math.Sqrt(filtVar[t]):F2})");
}

// --- Metriques : MSE filtree vs MSE brute (par rapport a la VRAIE trajectoire) ---
double rawMSE = 0.0, filtMSE = 0.0;
for (int t = 0; t < T; t++) {
    rawMSE  += Math.Pow(obs[t]      - trueState[t], 2);
    filtMSE += Math.Pow(filtMean[t] - trueState[t], 2);
}
rawMSE  /= T;
filtMSE /= T;

Console.WriteLine("\n=== Performance du filtre (par rapport a la trajectoire VRAIE) ===");
Console.WriteLine($"MSE observations brutes : {rawMSE:F3}   (variance de capteur R = {obsVar:F1})");
Console.WriteLine($"MSE estimation filtree  : {filtMSE:F3}");
Console.WriteLine($"Reduction d'erreur      : {(1.0 - filtMSE / rawMSE) * 100:F1}%   (filtre vs brut)");
Console.WriteLine($"Variance post. finale   : {filtVar[T - 1]:F3}   (vs R = {obsVar:F1})");


Echelle : -6,4 (gauche) -> 18,9 (droite), 60 colonnes


  VRAI = |   OBS = .   FILT = #


t= 0 VRAI |||||||||||||||


     OBS  .....


     FILT #####  (+-1,79)


...


=== Performance du filtre (par rapport a la trajectoire VRAIE) ===


MSE observations brutes : 4,875   (variance de capteur R = 4,0)


MSE estimation filtree  : 1,283


Reduction d'erreur      : 73,7%   (filtre vs brut)


Variance post. finale   : 1,186   (vs R = 4,0)


### Lecture du résultat

Le filtre **réduit fortement l'erreur** : la MSE filtrée est nettement inférieure à la MSE
des observations brutes. Deux mécanismes sont visibles dans le graphe ASCII :

- **Lissage** — l'estimation filtrée (`#`) varie beaucoup moins que les observations (`.`)
  d'un pas à l'autre : le filtre fait confiance à la dynamique ($Q$ petit) pour rejeter le
  bruit du capteur ($R$ grand).
- **Incertitude bornée** — la variance postérieure (±σ) se stabilise rapidement bien sous
  $R$ : après une brève phase d'initialisation, le filtre « sait » où il en est, et cette
  connaissance ne se dégrade pas avec le temps (équation de Riccati discrète).

C'est précisément la situation où le filtre de Kalman *aide* : capteur bruité, dynamique
fiable. L'exercice 1 explore la situation limite opposée (dynamique très incertaine).


## 4. Pourquoi ça marche : conjugaison exacte et borne de variance

Le filtre de Kalman tire sa puissance d'une propriété **exacte** : tout étant gaussien et
linéaire, **chaque postérieur est gaussien et calculable en forme fermée**. Infer.NET (EP)
retrouve cette solution exacte — c'est le cas où l'inférence est *exacte*, pas approchée.

La **règle de combinaison** est elle-même fermée : le postérieur $p(x_t \mid y_t)$ est
gaussien, avec une moyenne qui est un **pondéré** entre la prédiction (où le mobile
*devrait* être d'après la dynamique) et l'observation (où le capteur le *voit*). Le poids
relatif est l'inverse des variances : on fait davantage confiance à la source la plus
certaine.

- Si le capteur est **très bruité** ($R$ grand), le filtre fait confiance à la dynamique →
  lissage agressif, MSE fortement réduite (notre cas : $R = 4 \gg Q = 0{,}5$).
- Si la dynamique est **très incertaine** ($Q$ grand), le filtre fait confiance au capteur →
  l'estimation colle aux observations, peu de lissage (exercice 1).

La **borne de variance** (équation de Riccati) garantit que l'incertitude résiduelle
*plafonne* plutôt que d'exploser — c'est ce qui rend le filtre fiable sur le long terme.


## 5. Exercices

### Exercice 1 — Sensibilité au bruit de dynamique (Q)
Rejouez le filtre en balayant $Q \in \{0{,}01\,;\, 0{,}5\,;\, 2{,}0\,;\, 10{,}0\}$ (sans
changer la vraie trajectoire ni $R$). Pour chaque $Q$, relevez la **MSE filtrée** et la
**variance postérieure moyenne**. Observez le compromis : $Q$ petit → lissage agressif
(le filtre ignore le capteur et suit la pente) ; $Q$ grand → le filtre colle au capteur
(réagit au bruit). Quel $Q$ minimise la MSE sur **cette** trajectoire ?
- **Indice :** le $Q$ optimal est proche de la *vraie* variance de process (ici $0{,}5$).
  Choisir $Q$, c'est donc faire de l'**estimation de paramètre** — le filtre suppose connu
  ce que vous fixez à la main.


In [5]:
// Exercice 1 a completer
// Conseil : bouclez sur les valeurs de Q, relancez la recursion (copiez la boucle ci-dessus
// en parametrant processVar), stockez filtMSE pour chaque Q. Affichez le tableau Q|MSE|var.
Console.WriteLine("Exercice 1 a completer");


Exercice 1 a completer


### Exercice 2 — État vectoriel (position + vitesse)
Le filtre ci-dessus suit une position scalaire avec une *vitesse* (`drift`) injectée
manuellement. Généralisez à un état **vectoriel** $x = (\text{position}, \text{vitesse})$
où $\text{pos}_t = \text{pos}_{t-1} + \text{vit}_{t-1} \cdot \Delta t + \text{bruit}$ et
$\text{vit}_t = \text{vit}_{t-1} + \text{bruit}$. Utilisez `Variable.Vector` et
`Variable.MatrixTimesVector` d'Infer.NET.
- **Indice :** c'est le « vrai » filtre de Kalman matriciel. La **vitesse devient
  latente et estimée** (plus injectée). On observe uniquement la position
  ($y = \text{pos} + \text{bruit}$) et on infère la vitesse cachée — c'est tout l'intérêt.


In [6]:
// Exercice 2 a completer
// Conseil : etat x = Variable.Vector, transition par MatrixTimesVector(A, x[t-1]).
// Commencez par estimer la vitesse cachee a partir de observations de position bruitees.
Console.WriteLine("Exercice 2 a completer");


Exercice 2 a completer


### Exercice 3 — Lissage joint (full factor graph)
Le filtre ci-dessus est **séquentiel** (un pas à la fois : il n'utilise que les
observations passées). Construisez la version **jointe** : un seul
`VariableArray<double> x` sur un `Range` temporel, avec
`x[t] = Variable.GaussianFromMeanAndVariance(x[t - 1] + drift, processVar)` dans un
bloc `Variable.ForEach` (**auto-référence** `x[t-1]`), toutes les observations branchées,
et laissez EP inférer **toutes les marginales simultanément** — c'est le **lissage**
(utilise aussi les observations futures).
- **Indice :** le lissage rétro-propage l'information, donc **MSE lissée $\leq$ MSE filtrée**.
  La syntaxe d'auto-référence `x[t-1]` dans `Variable.ForEach` est délicate (cas `t == 0`
  à isoler avec `Variable.If`) — référez-vous aux exemples de chaînes gaussiennes Infer.NET.


In [7]:
// Exercice 3 a completer
// Conseil : Range t = new Range(T); VariableArray<double> x = Variable.Array<double>(t);
// Dans Variable.ForEach(t) : If(t==0) x[t]=prior; If(t>0) x[t]=Gaussian(x[t-1]+drift, Q).
// Comparez la MSE lisse (jointe) a la MSE filtree (sequentielle ci-dessus).
Console.WriteLine("Exercice 3 a completer");


Exercice 3 a completer


## 6. Ponts avec la série

| Direction | Lien | Relation |
|-----------|------|----------|
| ↔ Infer-11 | [Infer-11 — Séquences (HMM)](Infer-14-Sequences.ipynb) | HMM à état **discret** ↔ filtre de Kalman à état **continu** : même squelette markovien (état caché + observation), nature de l'état différente |
| ← Infer-2 | [Infer-2 — Gaussian Mixtures](Infer-2-Gaussian-Mixtures.ipynb) | Le Kalman repose sur la **conjugaison gaussienne** — Infer-2 en pose les bases |
| ↔ Infer-16 | [Infer-16 — Sparse GP](Infer-16-Sparse-Gaussian-Process.ipynb) | GP et Kalman modélisent tous deux de l'aléatoire structuré ; GP = espace fonctionnel **non-paramétrique**, Kalman = état fini **paramétrique** |

**Référence fondatrice** : Kalman, R. E. (1960) — *A New Approach to Linear Filtering and Prediction Problems*, ASME Journal of Basic Engineering 82(1). L'article fondateur du filtre de Kalman — l'algorithme d'estimation le plus répandu en pratique.

## 7. Conclusion

Le **filtre de Kalman** est le pont entre le HMM discret d'[Infer-11](Infer-14-Sequences.ipynb)
et le monde continu. Sa puissance tient en trois propriétés : pour les systèmes
**linéaires gaussiens**, l'inférence est **exacte** (conjugaison), **récursive** (un pas à
la fois, en $O(T)$) et **bornée** (variance postérieure stable). Infer.NET la résout
naturellement via EP, qui retrouve la solution exacte sur ce cas d'école.

La leçon générale : quand le modèle est **linéaire-gaussien**, nul besoin de méthodes
d'inférence approchées lourdes — la conjugaison offre une solution fermée. C'est en
**sortant** de ce régime (non-linéarités, distributions non gaussiennes) que les processus
gaussiens ([Infer-16](Infer-16-Sparse-Gaussian-Process.ipynb)) et les méthodes
variationnelles deviennent nécessaires. Le Kalman n'est pas un cas particulier encombrant :
c'est le **point de référence** par rapport auquel tous les filtres approchés se mesurent.